In [3]:
import os

MODE = "glue"

# S3 bucket names
S3_RAW_BUCKET       = os.environ.get("S3_RAW_BUCKET", "f1-pipeline-raw-layer-034381339055")
S3_PROCESSED_BUCKET = os.environ.get("S3_PROCESSED_BUCKET", "f1-pipeline-processed-layer-034381339055")
AWS_REGION          = os.environ.get("AWS_REGION", "ap-south-1")

RAW_BASE       = f"s3://{S3_RAW_BUCKET}/raw/jolpica"
PROCESSED_BASE = f"s3://{S3_PROCESSED_BUCKET}/processed/jolpica"

# Manifest storage
MANIFEST_BUCKET = S3_RAW_BUCKET
MANIFEST_PREFIX = "meta/manifests"

print(f"MODE:      {MODE}")
print(f"RAW:       {RAW_BASE}")
print(f"PROCESSED: {PROCESSED_BASE}")

MODE:      glue
RAW:       s3://f1-pipeline-raw-layer-034381339055/raw/jolpica
PROCESSED: s3://f1-pipeline-processed-layer-034381339055/processed/jolpica


In [4]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

# Initialize Glue context and Spark session
sc          = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark       = glueContext.spark_session

job = Job(glueContext)
try:
    args = getResolvedOptions(sys.argv, ["JOB_NAME"])
    job.init(args["JOB_NAME"], args)
except Exception:
    job.init("f1-jolpica-clean", {})

spark.sparkContext.setLogLevel("WARN")
print("Spark & Glue session ready.")

Spark & Glue session ready.


In [5]:
import boto3
from botocore.exceptions import ClientError
import json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, DoubleType, StringType

_manifest_s3 = None

def _get_manifest_s3():
    global _manifest_s3
    if _manifest_s3 is None:
        _manifest_s3 = boto3.client("s3")
    return _manifest_s3

def load_manifest(source):
    key = f"{MANIFEST_PREFIX}/{source}_manifest.json"
    try:
        obj = _get_manifest_s3().get_object(Bucket=MANIFEST_BUCKET, Key=key)
        return json.loads(obj["Body"].read())
    except ClientError as e:
        if e.response["Error"]["Code"] == "NoSuchKey":
            return {"last_updated": None, "processed": {}}
        raise

def save_manifest(source, manifest):
    manifest["last_updated"] = datetime.now(timezone.utc).isoformat()
    key = f"{MANIFEST_PREFIX}/{source}_manifest.json"
    _get_manifest_s3().put_object(
        Bucket=MANIFEST_BUCKET, Key=key,
        Body=json.dumps(manifest, indent=2), ContentType="application/json",
    )

def mark_file_done(s3_path, manifest):
    manifest["processed"][s3_path] = datetime.now(timezone.utc).isoformat()

def get_unprocessed(source, all_raw_paths):
    manifest = load_manifest(source)
    already = set(manifest["processed"].keys())
    unprocessed = [p for p in all_raw_paths if p not in already]
    print(f"  {source}: {len(already)} already processed, {len(unprocessed)} new")
    return unprocessed, manifest

@F.udf(LongType())
def parse_lap_time_ms(time_str):
    if time_str is None: return None
    try:
        if ":" in time_str:
            parts = time_str.split(":")
            minutes = int(parts[0])
            sec_parts = parts[1].split(".")
            seconds = int(sec_parts[0])
            millis = int(sec_parts[1]) if len(sec_parts) > 1 else 0
            return minutes * 60_000 + seconds * 1_000 + millis
        else:
            sec_parts = time_str.split(".")
            seconds = int(sec_parts[0])
            millis = int(sec_parts[1]) if len(sec_parts) > 1 else 0
            return seconds * 1_000 + millis
    except Exception:
        return None

def discover_round_paths():
    paths, s3, seen = [], boto3.client("s3"), set()
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=MANIFEST_BUCKET, Prefix="raw/jolpica/seasons/"):
        for obj in page.get("Contents", []):
            parts = obj["Key"].split("/")
            if len(parts) < 5: continue
            year_str, round_dir = parts[3], parts[4]
            if not year_str.isdigit() or not round_dir.startswith("round_"): continue
            key = (year_str, round_dir)
            if key not in seen:
                seen.add(key)
                round_num = int(round_dir.replace("round_", ""))
                paths.append({
                    "season": int(year_str),
                    "round":  round_num,
                    "path":   f"{RAW_BASE}/seasons/{year_str}/{round_dir}",
                })
    paths.sort(key=lambda x: (x["season"], x["round"]))
    return paths

def safe_read_json(path):
    try:
        df = spark.read.json(path, multiLine=True)
        if not df.rdd.isEmpty():
            return df
        return None
    except Exception:
        return None

In [6]:
def flatten_results(df_raw, season, round_num):
    try:
        races = df_raw.select(F.explode("MRData.RaceTable.Races").alias("race"))
        results = races.select(
            F.lit(season).alias("season"), F.lit(round_num).alias("round"),
            F.col("race.raceName").alias("race_name"),
            F.explode("race.Results").alias("r"),
        )
        base_cols = [
            F.col("season").cast(IntegerType()), F.col("round").cast(IntegerType()),
            F.col("race_name"),
            F.col("r.number").cast(IntegerType()).alias("driver_number"),
            F.col("r.position").cast(IntegerType()).alias("position"),
            F.col("r.positionText").alias("position_text"),
            F.col("r.points").cast(DoubleType()).alias("points"),
            F.col("r.Driver.driverId").alias("driver_id"),
            F.col("r.Driver.code").alias("driver_code"),
            F.col("r.Driver.permanentNumber").cast(IntegerType()).alias("permanent_number"),
            F.col("r.Driver.givenName").alias("forename"),
            F.col("r.Driver.familyName").alias("surname"),
            F.col("r.Constructor.constructorId").alias("constructor_id"),
            F.col("r.Constructor.name").alias("constructor_name"),
            F.col("r.grid").cast(IntegerType()).alias("grid"),
            F.col("r.laps").cast(IntegerType()).alias("laps"),
            F.col("r.status").alias("status"),
            F.col("r.Time.millis").cast(LongType()).alias("time_millis"),
            F.col("r.Time.time").alias("time_text"),
            F.col("r.FastestLap.rank").cast(IntegerType()).alias("fastest_lap_rank"),
            F.col("r.FastestLap.lap").cast(IntegerType()).alias("fastest_lap_number"),
            parse_lap_time_ms(F.col("r.FastestLap.Time.time")).alias("fastest_lap_time_ms"),
        ]
        try:
            flat = results.select(*base_cols, 
                F.col("r.FastestLap.AverageSpeed.speed").cast(DoubleType()).alias("fastest_lap_speed")
            ).dropDuplicates()
        except Exception:
            flat = results.select(*base_cols, 
                F.lit(None).cast(DoubleType()).alias("fastest_lap_speed")
            ).dropDuplicates()
        return flat
    except Exception as e:
        print(f"    ⚠ Could not flatten results for {season}/R{round_num:02d}: {e}")
        return None

def flatten_qualifying(df_raw, season, round_num):
    try:
        races = df_raw.select(F.explode("MRData.RaceTable.Races").alias("race"))
        quali = races.select(
            F.lit(season).alias("season"), F.lit(round_num).alias("round"),
            F.explode("race.QualifyingResults").alias("q"),
        )
        return quali.select(
            F.col("season").cast(IntegerType()), F.col("round").cast(IntegerType()),
            F.col("q.number").cast(IntegerType()).alias("driver_number"),
            F.col("q.position").cast(IntegerType()).alias("position"),
            F.col("q.Driver.driverId").alias("driver_id"),
            F.col("q.Driver.code").alias("driver_code"),
            F.col("q.Constructor.constructorId").alias("constructor_id"),
            F.col("q.Q1").alias("q1"), F.col("q.Q2").alias("q2"), F.col("q.Q3").alias("q3"),
            parse_lap_time_ms(F.col("q.Q1")).alias("q1_ms"),
            parse_lap_time_ms(F.col("q.Q2")).alias("q2_ms"),
            parse_lap_time_ms(F.col("q.Q3")).alias("q3_ms"),
        ).dropDuplicates()
    except Exception as e:
        print(f"    ⚠ Could not flatten qualifying for {season}/R{round_num:02d}: {e}")
        return None

def flatten_pitstops(df_raw, season, round_num):
    try:
        races = df_raw.select(F.explode("MRData.RaceTable.Races").alias("race"))
        pits = races.select(
            F.lit(season).alias("season"), F.lit(round_num).alias("round"),
            F.explode("race.PitStops").alias("p"),
        )
        return pits.select(
            F.col("season").cast(IntegerType()), F.col("round").cast(IntegerType()),
            F.col("p.driverId").alias("driver_id"),
            F.col("p.stop").cast(IntegerType()).alias("stop"),
            F.col("p.lap").cast(IntegerType()).alias("lap"),
            F.col("p.time").alias("time"),
            F.col("p.duration").cast(DoubleType()).alias("duration_seconds"),
            (F.col("p.duration").cast(DoubleType()) * 1000).cast(LongType()).alias("duration_ms"),
        ).dropDuplicates()
    except Exception as e:
        print(f"    ⚠ Could not flatten pitstops for {season}/R{round_num:02d}: {e}")
        return None

def flatten_driver_standings(df_raw, season, round_num):
    try:
        standings = df_raw.select(F.explode("MRData.StandingsTable.StandingsLists").alias("sl"))
        drivers = standings.select(
            F.lit(season).alias("season"), F.lit(round_num).alias("round"),
            F.explode("sl.DriverStandings").alias("ds"),
        )
        return drivers.select(
            F.col("season").cast(IntegerType()), F.col("round").cast(IntegerType()),
            F.col("ds.Driver.driverId").alias("driver_id"),
            F.col("ds.Driver.code").alias("driver_code"),
            F.col("ds.Driver.givenName").alias("forename"),
            F.col("ds.Driver.familyName").alias("surname"),
            F.col("ds.Constructors")[0]["constructorId"].alias("constructor_id"),
            F.col("ds.position").cast(IntegerType()).alias("position"),
            F.col("ds.positionText").alias("position_text"),
            F.col("ds.points").cast(DoubleType()).alias("points"),
            F.col("ds.wins").cast(IntegerType()).alias("wins"),
        ).dropDuplicates()
    except Exception as e:
        print(f"    ⚠ Could not flatten driver_standings for {season}/R{round_num:02d}: {e}")
        return None

def flatten_constructor_standings(df_raw, season, round_num):
    try:
        standings = df_raw.select(F.explode("MRData.StandingsTable.StandingsLists").alias("sl"))
        constructors = standings.select(
            F.lit(season).alias("season"), F.lit(round_num).alias("round"),
            F.explode("sl.ConstructorStandings").alias("cs"),
        )
        return constructors.select(
            F.col("season").cast(IntegerType()), F.col("round").cast(IntegerType()),
            F.col("cs.Constructor.constructorId").alias("constructor_id"),
            F.col("cs.Constructor.name").alias("constructor_name"),
            F.col("cs.Constructor.nationality").alias("nationality"),
            F.col("cs.position").cast(IntegerType()).alias("position"),
            F.col("cs.positionText").alias("position_text"),
            F.col("cs.points").cast(DoubleType()).alias("points"),
            F.col("cs.wins").cast(IntegerType()).alias("wins"),
        ).dropDuplicates()
    except Exception as e:
        print(f"    ⚠ Could not flatten constructor_standings for {season}/R{round_num:02d}: {e}")
        return None

def flatten_sprint(df_raw, season, round_num):
    try:
        races = df_raw.select(F.explode("MRData.RaceTable.Races").alias("race"))
        try:
            results = races.select(
                F.lit(season).alias("season"), F.lit(round_num).alias("round"),
                F.explode("race.SprintResults").alias("r"), 
            )
        except Exception:
            return None # Silently skip non-sprint weekends

        flat = results.select(
            F.col("season").cast(IntegerType()), F.col("round").cast(IntegerType()),
            F.col("r.number").cast(IntegerType()).alias("driver_number"),
            F.col("r.position").cast(IntegerType()).alias("position"),
            F.col("r.positionText").alias("position_text"),
            F.col("r.points").cast(DoubleType()).alias("points"),
            F.col("r.Driver.driverId").alias("driver_id"),
            F.col("r.Driver.code").alias("driver_code"),
            F.col("r.Constructor.constructorId").alias("constructor_id"),
            F.col("r.Constructor.name").alias("constructor_name"),
            F.col("r.grid").cast(IntegerType()).alias("grid"),
            F.col("r.laps").cast(IntegerType()).alias("laps"),
            F.col("r.status").alias("status"),
            F.col("r.Time.millis").cast(LongType()).alias("time_millis"),
            F.col("r.Time.time").alias("time_text"),
        ).dropDuplicates()
        return flat
    except Exception as e:
        print(f"    ⚠ Could not flatten sprint for {season}/R{round_num:02d}: {e}")
        return None

In [7]:
ENDPOINT_CONFIG = {
    "results":                 (flatten_results,                "results.json"),
    "qualifying":              (flatten_qualifying,             "qualifying.json"),
    "pitstops":                (flatten_pitstops,               "pitstops.json"),
    "driver_standings":        (flatten_driver_standings,       "driver_standings.json"),
    "constructor_standings":   (flatten_constructor_standings,  "constructor_standings.json"),
    "sprint":                  (flatten_sprint,                 "sprint.json"),
}

print("=" * 60)
print("clean_jolpica.py — Starting")
print("=" * 60)

round_paths = discover_round_paths()
print(f"  Found {len(round_paths)} round paths to scan\n")

ALL_RAW_PATHS = []
for rp in round_paths:
    for _, (_, filename) in ENDPOINT_CONFIG.items():
        ALL_RAW_PATHS.append(
            f"raw/jolpica/seasons/{rp['season']}/round_{rp['round']:02d}/{filename}"
        )

unprocessed, manifest = get_unprocessed("jolpica", ALL_RAW_PATHS)

if not unprocessed:
    print("All files already processed — nothing to do.")
else:
    for raw_path in unprocessed:
        parts = raw_path.split("/")
        season    = int(parts[3])
        round_dir = parts[4]
        round_num = int(round_dir.replace("round_", ""))
        filename  = parts[5]

        flatten_fn    = None
        endpoint_name = None
        for ep_name, (fn, fn_name) in ENDPOINT_CONFIG.items():
            if fn_name == filename:
                flatten_fn    = fn
                endpoint_name = ep_name
                break

        if flatten_fn is None:
            print(f"  ⚠ Unknown file {filename} — skipping")
            continue

        json_path = f"{RAW_BASE}/seasons/{season}/{round_dir}/{filename}"
        df_raw = safe_read_json(json_path)

        if df_raw is None:
            continue

        df_clean = flatten_fn(df_raw, season, round_num)

        if df_clean is None or df_clean.rdd.isEmpty():
            continue

        # Write with coalesce(1) to prevent the small file problem
        path = f"{PROCESSED_BASE}/{endpoint_name}"
        df_clean.coalesce(1).write.mode("append").partitionBy("season").parquet(path)
        print(f"  ✓ {raw_path}")

        mark_file_done(raw_path, manifest)
        save_manifest("jolpica", manifest)

    print("\n" + "=" * 60)
    print("clean_jolpica.py — Complete ✓")
    print("=" * 60)

job.commit()

clean_jolpica.py — Starting
  Found 26 round paths to scan

  jolpica: 111 already processed, 45 new
  ✓ raw/jolpica/seasons/2025/round_01/results.json
  ✓ raw/jolpica/seasons/2025/round_02/results.json
  ✓ raw/jolpica/seasons/2025/round_03/results.json
  ✓ raw/jolpica/seasons/2025/round_04/results.json
  ✓ raw/jolpica/seasons/2025/round_05/results.json
  ✓ raw/jolpica/seasons/2025/round_06/results.json
  ✓ raw/jolpica/seasons/2025/round_07/results.json
  ✓ raw/jolpica/seasons/2025/round_08/results.json
  ✓ raw/jolpica/seasons/2025/round_09/results.json
  ✓ raw/jolpica/seasons/2025/round_10/results.json
  ✓ raw/jolpica/seasons/2025/round_11/results.json
  ✓ raw/jolpica/seasons/2025/round_12/results.json
  ✓ raw/jolpica/seasons/2025/round_13/results.json
  ✓ raw/jolpica/seasons/2025/round_14/results.json
  ✓ raw/jolpica/seasons/2025/round_15/results.json
  ✓ raw/jolpica/seasons/2025/round_16/results.json
  ✓ raw/jolpica/seasons/2025/round_17/results.json
  ✓ raw/jolpica/seasons/2025/rou